# **Finetuning Loop - 1**

In [ ]:
# ===============================
# 1. Install & Import Libraries
# ===============================
!pip install -q transformers scikit-learn pandas tqdm

import os, json, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# ===============================
# 2. Configuration
# ===============================
DATA_PATH = "/content/hr_intents_dataset_expanded.csv"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "./multihead_hr_model"
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5
MAX_LEN = 128
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

# ===============================
# 3. Load & Preprocess Dataset
# ===============================
df = pd.read_csv(DATA_PATH).dropna()

intent_encoder = LabelEncoder()
task_encoder = LabelEncoder()
df["intent_label"] = intent_encoder.fit_transform(df["intent"])
df["task_label"] = task_encoder.fit_transform(df["task"])

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(f"{OUTPUT_DIR}/intent_mapping.json", "w") as f:
    json.dump({i: label for i, label in enumerate(intent_encoder.classes_)}, f)
with open(f"{OUTPUT_DIR}/task_mapping.json", "w") as f:
    json.dump({i: label for i, label in enumerate(task_encoder.classes_)}, f)

# ===============================
# 4. Initialize Tokenizer
# ===============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ===============================
# 5. BIO Tagging for Slots
# ===============================
def bio_tagging(text, slot_json):
    tokens = tokenizer.tokenize(text)
    labels = ["O"] * len(tokens)
    for key, value in slot_json.items():
        if not value:
            continue
        value_tokens = tokenizer.tokenize(str(value))
        for i in range(len(tokens) - len(value_tokens) + 1):
            if tokens[i:i+len(value_tokens)] == value_tokens:
                labels[i] = f"B-{key}"
                for j in range(1, len(value_tokens)):
                    labels[i+j] = f"I-{key}"
                break
    return tokens, labels

slot_labels_set = set()
bio_labels = []
for _, row in df.iterrows():
    slot_dict = json.loads(row["slots"])
    _, labels = bio_tagging(row["text"], slot_dict)
    bio_labels.append(labels)
    slot_labels_set.update(labels)

slot_label_list = sorted(list(slot_labels_set))
slot_label_map = {label: i for i, label in enumerate(slot_label_list)}
slot_ignore_index = slot_label_map["O"] # Use 'O' label as ignore index for padding

with open(f"{OUTPUT_DIR}/slot_label_mapping.json", "w") as f:
    json.dump(slot_label_map, f)

# ===============================
# 6. Dataset Class
# ===============================
class HRMultiHeadSlotDataset(Dataset):
    def __init__(self, df, bio_labels, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.intent_labels = torch.tensor(df["intent_label"].tolist())
        self.task_labels = torch.tensor(df["task_label"].tolist())
        self.encodings = tokenizer(self.texts, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
        self.slot_labels = []
        for i, labels in enumerate(bio_labels):
            label_ids = [slot_label_map.get(l, slot_label_map["O"]) for l in labels]
            # Truncate or pad slot labels to match the tokenized input length after padding
            label_ids = label_ids[:max_len] + [slot_ignore_index] * (max_len - len(label_ids))
            self.slot_labels.append(torch.tensor(label_ids))

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["intent_labels"] = self.intent_labels[idx]
        item["task_labels"] = self.task_labels[idx]
        item["slot_labels"] = self.slot_labels[idx]
        return item

    def __len__(self):
        return len(self.intent_labels)

# ===============================
# 7. Multi-Head Model Definition
# ===============================
class MultiHeadClassifier(torch.nn.Module):
    def __init__(self, base_model_name, num_intents, num_tasks, num_slot_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.encoder.config.hidden_size
        self.intent_head = torch.nn.Linear(hidden_size, num_intents)
        self.task_head = torch.nn.Linear(hidden_size, num_tasks)
        self.slot_head = torch.nn.Linear(hidden_size, num_slot_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, intent_labels=None, task_labels=None, slot_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.last_hidden_state[:, 0]
        intent_logits = self.intent_head(pooled_output)
        task_logits = self.task_head(pooled_output)
        slot_logits = self.slot_head(outputs.last_hidden_state)

        loss = None
        if intent_labels is not None and task_labels is not None and slot_labels is not None:
            intent_loss = F.cross_entropy(intent_logits, intent_labels)
            task_loss = F.cross_entropy(task_logits, task_labels)
            # Flatten slot logits and labels, and ignore loss for padding tokens
            slot_loss = F.cross_entropy(slot_logits.view(-1, slot_logits.shape[-1]), slot_labels.view(-1), ignore_index=slot_ignore_index)
            loss = intent_loss + task_loss + slot_loss

        return {
            "loss": loss,
            "intent_logits": intent_logits,
            "task_logits": task_logits,
            "slot_logits": slot_logits
        }

# ===============================
# 8. Training Setup
# ===============================
model = MultiHeadClassifier(MODEL_NAME, len(intent_encoder.classes_), len(task_encoder.classes_), len(slot_label_list))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_df, val_df, train_bio, val_bio = train_test_split(df, bio_labels, test_size=0.2, random_state=42, stratify=df["intent_label"])
train_dataset = HRMultiHeadSlotDataset(train_df, train_bio, tokenizer, MAX_LEN)
val_dataset = HRMultiHeadSlotDataset(val_df, val_bio, tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps)

# ===============================
# 9. Training Loop
# ===============================
for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        loop.set_postfix(loss=loss.item())

    # Validation
    model.eval()
    all_intent_preds, all_task_preds = [], []
    all_intent_labels, all_task_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], token_type_ids=batch.get("token_type_ids"))
            intent_preds = torch.argmax(outputs["intent_logits"], dim=-1)
            task_preds = torch.argmax(outputs["task_logits"], dim=-1)
            all_intent_preds.extend(intent_preds.cpu().numpy())
            all_task_preds.extend(task_preds.cpu().numpy())
            all_intent_labels.extend(batch["intent_labels"].cpu().numpy())
            all_task_labels.extend(batch["task_labels"].cpu().numpy())

    intent_acc = accuracy_score(all_intent_labels, all_intent_preds)
    task_acc = accuracy_score(all_task_labels, all_task_preds)
    intent_f1 = f1_score(all_intent_labels, all_intent_preds, average="weighted")
    task_f1 = f1_score(all_task_labels, all_task_preds, average="weighted")
    print(f"Epoch {epoch+1} | Intent Acc: {intent_acc:.4f} | Task Acc: {task_acc:.4f} | Intent F1: {intent_f1:.4f} | Task F1: {task_f1:.4f}")

# ===============================
# 10. Save Model
# ===============================
model.encoder.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
torch.save(model.state_dict(), f"{OUTPUT_DIR}/multihead_model.pt")
print(f"\n✅ Multi-head model with slot extraction saved at: {OUTPUT_DIR}")

Epoch 1 | Intent Acc: 0.9091 | Task Acc: 0.5705 | Intent F1: 0.9079 | Task F1: 0.4904


Epoch 2 | Intent Acc: 0.9795 | Task Acc: 0.8818 | Intent F1: 0.9794 | Task F1: 0.8828


Epoch 3 | Intent Acc: 0.9841 | Task Acc: 0.9068 | Intent F1: 0.9840 | Task F1: 0.9072


Epoch 4 | Intent Acc: 0.9841 | Task Acc: 0.9136 | Intent F1: 0.9840 | Task F1: 0.9142


Epoch 5 | Intent Acc: 0.9864 | Task Acc: 0.9227 | Intent F1: 0.9863 | Task F1: 0.9234


Epoch 6 | Intent Acc: 0.9886 | Task Acc: 0.9432 | Intent F1: 0.9886 | Task F1: 0.9433


Epoch 7 | Intent Acc: 0.9886 | Task Acc: 0.9432 | Intent F1: 0.9886 | Task F1: 0.9432


Epoch 8 | Intent Acc: 0.9886 | Task Acc: 0.9500 | Intent F1: 0.9886 | Task F1: 0.9501


Epoch 9 | Intent Acc: 0.9909 | Task Acc: 0.9545 | Intent F1: 0.9909 | Task F1: 0.9546


Epoch 10 | Intent Acc: 0.9909 | Task Acc: 0.9545 | Intent F1: 0.9909 | Task F1: 0.9546

✅ Multi-head model with slot extraction saved at: ./multihead_hr_model


## **Inference**

In [ ]:
import torch, json, time
from transformers import AutoTokenizer, AutoModel

MODEL_PATH = "/content/multihead_hr_model"

# Load mappings
with open(f"{MODEL_PATH}/intent_mapping.json") as f:
    intent_map = json.load(f)
with open(f"{MODEL_PATH}/task_mapping.json") as f:
    task_map = json.load(f)
with open(f"{MODEL_PATH}/slot_label_mapping.json") as f:
    slot_map = json.load(f)
slot_map_rev = {v: k for k, v in slot_map.items()}

# Load tokenizer and encoder
tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
encoder = AutoModel.from_pretrained(MODEL_PATH)

# Define model
class MultiHeadClassifier(torch.nn.Module):
    def __init__(self, encoder, num_intents, num_tasks, num_slot_labels):
        super().__init__()
        self.encoder = encoder
        hidden_size = encoder.config.hidden_size
        self.intent_head = torch.nn.Linear(hidden_size, num_intents)
        self.task_head = torch.nn.Linear(hidden_size, num_tasks)
        self.slot_head = torch.nn.Linear(hidden_size, num_slot_labels)

    def forward(self, input_ids, attention_mask):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled_output = outputs.last_hidden_state[:, 0]
        intent_logits = self.intent_head(pooled_output)
        task_logits = self.task_head(pooled_output)
        slot_logits = self.slot_head(outputs.last_hidden_state)
        return intent_logits, task_logits, slot_logits

# Load model weights
model = MultiHeadClassifier(encoder, len(intent_map), len(task_map), len(slot_map))
model.load_state_dict(torch.load(f"{MODEL_PATH}/multihead_model.pt", map_location=torch.device("cpu")))
model.eval()

# Decode BIO tags using word_ids
def decode_slots(text, slot_ids, input_tokens, word_ids):
    slots = {}
    current_slot = None
    current_value = []
    for idx, slot_id in enumerate(slot_ids):
        label = slot_map_rev.get(slot_id, "O")
        word_idx = word_ids[idx]
        if word_idx is None or input_tokens[idx] in ["[CLS]", "[SEP]"]:
            continue
        token = input_tokens[idx]
        if label.startswith("B-"):
            if current_slot:
                slots[current_slot] = tokenizer.convert_tokens_to_string(current_value)
            current_slot = label[2:]
            current_value = [token]
        elif label.startswith("I-") and current_slot == label[2:]:
            current_value.append(token)
        else:
            if current_slot:
                slots[current_slot] = tokenizer.convert_tokens_to_string(current_value)
            current_slot = None
            current_value = []
    if current_slot:
        slots[current_slot] = tokenizer.convert_tokens_to_string(current_value)
    return slots

# Inference function
def predict(text, role="employee"):
    start = time.time()
    encoding = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    input_ids = encoding["input_ids"]
    attention_mask = encoding["attention_mask"]
    input_tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    word_ids = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128).word_ids()

    with torch.no_grad():
        intent_logits, task_logits, slot_logits = model(input_ids=input_ids, attention_mask=attention_mask)
        intent = intent_map[str(torch.argmax(intent_logits, dim=-1).item())]
        task = task_map[str(torch.argmax(task_logits, dim=-1).item())]
        slot_ids = torch.argmax(slot_logits, dim=-1).squeeze().tolist()
        slots = decode_slots(text, slot_ids, input_tokens, word_ids)

    end = time.time()
    print(f"Inference time: {end - start:.3f} seconds")
    return {
        "intent": intent,
        "task": task,
        "slots": slots,
        "role": role
    }

# Example
query = "Can you provide the holiday list for 2025?"
result = predict(query)
print(result)


Inference time: 0.017 seconds
{'intent': 'holiday_calendar', 'task': 'holiday_list_year', 'slots': {'year': '?'}, 'role': 'employee'}


# **Finetuning Loop - 2**

In [ ]:
# ===============================
# 1. Install & Import Libraries
# ===============================
!pip install -q transformers scikit-learn pandas tqdm

import os, json, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# ===============================
# 2. Configuration
# ===============================
DATA_PATH = "/content/hr_intents_dataset_expanded.csv"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "./multihead_hr_model"
BATCH_SIZE = 16
EPOCHS = 10
LEARNING_RATE = 2e-5
MAX_LEN = 128
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

# ===============================
# 3. Load & Preprocess Dataset
# ===============================
df = pd.read_csv(DATA_PATH).dropna()

intent_encoder = LabelEncoder()
task_encoder = LabelEncoder()
df["intent_label"] = intent_encoder.fit_transform(df["intent"])
df["task_label"] = task_encoder.fit_transform(df["task"])

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(f"{OUTPUT_DIR}/intent_mapping.json", "w") as f:
    json.dump({i: label for i, label in enumerate(intent_encoder.classes_)}, f)
with open(f"{OUTPUT_DIR}/task_mapping.json", "w") as f:
    json.dump({i: label for i, label in enumerate(task_encoder.classes_)}, f)

# ===============================
# 4. Initialize Tokenizer
# ===============================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

# ===============================
# 5. BIO Tagging for Slots
# ===============================
def bio_tagging(text, slot_json):
    tokens = tokenizer.tokenize(text)
    labels = ["O"] * len(tokens)
    for key, value in slot_json.items():
        if not value:
            continue
        value_tokens = tokenizer.tokenize(str(value))
        for i in range(len(tokens) - len(value_tokens) + 1):
            if tokens[i:i+len(value_tokens)] == value_tokens:
                labels[i] = f"B-{key}"
                for j in range(1, len(value_tokens)):
                    labels[i+j] = f"I-{key}"
                break
    return tokens, labels

slot_labels_set = set()
bio_labels = []
for _, row in df.iterrows():
    slot_dict = json.loads(row["slots"])
    _, labels = bio_tagging(row["text"], slot_dict)
    bio_labels.append(labels)
    slot_labels_set.update(labels)

slot_label_list = sorted(list(slot_labels_set))
slot_label_map = {label: i for i, label in enumerate(slot_label_list)}
slot_ignore_index = slot_label_map["O"] # Use 'O' label as ignore index for padding

with open(f"{OUTPUT_DIR}/slot_label_mapping.json", "w") as f:
    json.dump(slot_label_map, f)

# ===============================
# 6. Dataset Class
# ===============================
class HRMultiHeadSlotDataset(Dataset):
    def __init__(self, df, bio_labels, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.intent_labels = torch.tensor(df["intent_label"].tolist())
        self.task_labels = torch.tensor(df["task_label"].tolist())
        self.encodings = tokenizer(self.texts, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
        self.slot_labels = []
        for i, labels in enumerate(bio_labels):
            label_ids = [slot_label_map.get(l, slot_label_map["O"]) for l in labels]
            # Truncate or pad slot labels to match the tokenized input length after padding
            label_ids = label_ids[:max_len] + [slot_ignore_index] * (max_len - len(label_ids))
            self.slot_labels.append(torch.tensor(label_ids))

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["intent_labels"] = self.intent_labels[idx]
        item["task_labels"] = self.task_labels[idx]
        item["slot_labels"] = self.slot_labels[idx]
        return item

    def __len__(self):
        return len(self.intent_labels)

# ===============================
# 7. Multi-Head Model Definition
# ===============================
class MultiHeadClassifier(torch.nn.Module):
    def __init__(self, base_model_name, num_intents, num_tasks, num_slot_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.encoder.config.hidden_size
        self.intent_head = torch.nn.Linear(hidden_size, num_intents)
        self.task_head = torch.nn.Linear(hidden_size, num_tasks)
        self.slot_head = torch.nn.Linear(hidden_size, num_slot_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, intent_labels=None, task_labels=None, slot_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.last_hidden_state[:, 0]
        intent_logits = self.intent_head(pooled_output)
        task_logits = self.task_head(pooled_output)
        slot_logits = self.slot_head(outputs.last_hidden_state)

        loss = None
        if intent_labels is not None and task_labels is not None and slot_labels is not None:
            intent_loss = F.cross_entropy(intent_logits, intent_labels)
            task_loss = F.cross_entropy(task_logits, task_labels)
            # Flatten slot logits and labels, and ignore loss for padding tokens
            slot_loss = F.cross_entropy(slot_logits.view(-1, slot_logits.shape[-1]), slot_labels.view(-1), ignore_index=slot_ignore_index)
            loss = intent_loss + task_loss + slot_loss

        return {
            "loss": loss,
            "intent_logits": intent_logits,
            "task_logits": task_logits,
            "slot_logits": slot_logits
        }

# ===============================
# 8. Training Setup
# ===============================
model = MultiHeadClassifier(MODEL_NAME, len(intent_encoder.classes_), len(task_encoder.classes_), len(slot_label_list))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_df, val_df, train_bio, val_bio = train_test_split(df, bio_labels, test_size=0.2, random_state=42, stratify=df["intent_label"])
train_dataset = HRMultiHeadSlotDataset(train_df, train_bio, tokenizer, MAX_LEN)
val_dataset = HRMultiHeadSlotDataset(val_df, val_bio, tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps)

# ===============================
# 9. Training Loop
# ===============================
for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        loop.set_postfix(loss=loss.item())

    # Validation
    model.eval()
    all_intent_preds, all_task_preds = [], []
    all_intent_labels, all_task_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], token_type_ids=batch.get("token_type_ids"))
            intent_preds = torch.argmax(outputs["intent_logits"], dim=-1)
            task_preds = torch.argmax(outputs["task_logits"], dim=-1)
            all_intent_preds.extend(intent_preds.cpu().numpy())
            all_task_preds.extend(task_preds.cpu().numpy())
            all_intent_labels.extend(batch["intent_labels"].cpu().numpy())
            all_task_labels.extend(batch["task_labels"].cpu().numpy())

    intent_acc = accuracy_score(all_intent_labels, all_intent_preds)
    task_acc = accuracy_score(all_task_labels, all_task_preds)
    intent_f1 = f1_score(all_intent_labels, all_intent_preds, average="weighted")
    task_f1 = f1_score(all_task_labels, all_task_preds, average="weighted")
    print(f"Epoch {epoch+1} | Intent Acc: {intent_acc:.4f} | Task Acc: {task_acc:.4f} | Intent F1: {intent_f1:.4f} | Task F1: {task_f1:.4f}")

# ===============================
# 10. Save Model
# ===============================
model.encoder.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
torch.save(model.state_dict(), f"{OUTPUT_DIR}/multihead_model.pt")
print(f"\n✅ Multi-head model with slot extraction saved at: {OUTPUT_DIR}")

Epoch 1 | Intent Acc: 0.8727 | Task Acc: 0.6114 | Intent F1: 0.8715 | Task F1: 0.5444


Epoch 2 | Intent Acc: 0.9795 | Task Acc: 0.8682 | Intent F1: 0.9794 | Task F1: 0.8692


Epoch 3 | Intent Acc: 0.9864 | Task Acc: 0.9068 | Intent F1: 0.9864 | Task F1: 0.9084


Epoch 4 | Intent Acc: 0.9864 | Task Acc: 0.9159 | Intent F1: 0.9864 | Task F1: 0.9168


Epoch 5 | Intent Acc: 0.9886 | Task Acc: 0.9386 | Intent F1: 0.9886 | Task F1: 0.9388


Epoch 6 | Intent Acc: 0.9886 | Task Acc: 0.9432 | Intent F1: 0.9886 | Task F1: 0.9434


Epoch 7 | Intent Acc: 0.9909 | Task Acc: 0.9455 | Intent F1: 0.9909 | Task F1: 0.9459


Epoch 8 | Intent Acc: 0.9886 | Task Acc: 0.9500 | Intent F1: 0.9886 | Task F1: 0.9502


Epoch 9 | Intent Acc: 0.9909 | Task Acc: 0.9477 | Intent F1: 0.9909 | Task F1: 0.9479


Epoch 10 | Intent Acc: 0.9886 | Task Acc: 0.9545 | Intent F1: 0.9886 | Task F1: 0.9547

✅ Multi-head model with slot extraction saved at: ./multihead_hr_model


In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import json
import time
import numpy as np

# -------------------------
# Load Model & Tokenizer
# -------------------------
MODEL_PATH = "/content/multihead_hr_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
encoder = AutoModel.from_pretrained(MODEL_PATH)

# Load label mappings
with open(f"{MODEL_PATH}/intent_mapping.json") as f:
    intent_map = json.load(f)
with open(f"{MODEL_PATH}/task_mapping.json") as f:
    task_map = json.load(f)
with open(f"{MODEL_PATH}/slot_label_mapping.json") as f:
    slot_map = json.load(f)

# Reverse maps for decoding
intent_map_rev = {v: k for k, v in intent_map.items()}
task_map_rev = {v: k for k, v in task_map.items()}
slot_map_rev = {v: k for k, v in slot_map.items()}
slot_ignore_index = slot_map["O"]


# -------------------------
# Define Multi-Head Model
# -------------------------
class MultiHeadClassifier(torch.nn.Module):
    def __init__(self, encoder, num_intents, num_tasks, num_slot_labels):
        super().__init__()
        self.encoder = encoder
        hidden_size = encoder.config.hidden_size
        self.intent_head = torch.nn.Linear(hidden_size, num_intents)
        self.task_head = torch.nn.Linear(hidden_size, num_tasks)
        self.slot_head = torch.nn.Linear(hidden_size, num_slot_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.last_hidden_state[:, 0]
        intent_logits = self.intent_head(pooled_output)
        task_logits = self.task_head(pooled_output)
        slot_logits = self.slot_head(outputs.last_hidden_state)
        return intent_logits, task_logits, slot_logits

# -------------------------
# Load Model Weights
# -------------------------
model = MultiHeadClassifier(encoder, len(intent_map), len(task_map), len(slot_map))
model.load_state_dict(torch.load(f"{MODEL_PATH}/multihead_model.pt", map_location=torch.device("cpu")))
model.eval()

# -------------------------
# Decode BIO tags with Alignment
# -------------------------
def decode_slots(text, slot_ids, word_ids, tokens):
    slots = {}
    current_slot = None
    current_value = []
    last_word_id = None

    for idx, slot_id in enumerate(slot_ids):
        label = slot_map_rev.get(slot_id, "O")
        word_idx = word_ids[idx]
        token = tokens[idx]

        if word_idx is None or token in ["[CLS]", "[SEP]", "[PAD]"]:
            continue

        if label.startswith("B-"):
            if current_slot:
                slots[current_slot] = tokenizer.convert_tokens_to_string(current_value)
            current_slot = label[2:]
            current_value = [token]
            last_word_id = word_idx
        elif label.startswith("I-") and current_slot == label[2:] and word_idx == last_word_id:
             current_value.append(token)
        elif label.startswith("I-") and current_slot == label[2:] and word_idx != last_word_id:
            # This handles cases where the entity spans multiple words
            current_value.append(token)
            last_word_id = word_idx
        else:
            if current_slot:
                slots[current_slot] = tokenizer.convert_tokens_to_string(current_value)
            current_slot = None
            current_value = []
            last_word_id = None

    if current_slot:
        slots[current_slot] = tokenizer.convert_tokens_to_string(current_value)

    return slots


# -------------------------
# Inference Function
# -------------------------
def predict(text, role="employee"):
    start = time.time()
    encoding = tokenizer(text, return_tensors="pt", truncation=True, padding="max_length", max_length=128)
    word_ids = encoding.word_ids(batch_index=0)
    tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"][0])

    with torch.no_grad():
        intent_logits, task_logits, slot_logits = model(input_ids=encoding["input_ids"], attention_mask=encoding["attention_mask"])
        intent = intent_map[str(torch.argmax(intent_logits, dim=-1).item())]
        task = task_map[str(torch.argmax(task_logits, dim=-1).item())]
        slot_ids = torch.argmax(slot_logits, dim=-1).squeeze().tolist()

        slots = decode_slots(text, slot_ids, word_ids, tokens)

    end = time.time()
    print(f"Inference time: {end - start:.3f} seconds")
    return {
        "intent": intent,
        "task": task,
        "slots": slots,
        "role": role
    }

# -------------------------
# Example Usage
# -------------------------
query = "Can you provide the holiday list for 2025?"
result = predict(query)
print(result)

Inference time: 0.039 seconds
{'intent': 'holiday_calendar', 'task': 'holiday_list_year', 'slots': {'year': '?'}, 'role': 'employee'}


# **Finetuning - 3** Finalized

In [1]:
# 1. Install dependencies
!pip install -q transformers scikit-learn pandas tqdm

# 2. Import libraries
import os, json, pandas as pd, torch
from torch.utils.data import Dataset, DataLoader
from torch.nn import functional as F
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score
from tqdm import tqdm

# 3. Config
DATA_PATH = "/content/hr_intents_dataset_expanded.csv"
MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
OUTPUT_DIR = "./multihead_hr_model"
BATCH_SIZE = 16
EPOCHS = 50
LEARNING_RATE = 2e-5
MAX_LEN = 128
WARMUP_RATIO = 0.1
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0

# 4. Load dataset
df = pd.read_csv(DATA_PATH).dropna()
intent_encoder = LabelEncoder()
task_encoder = LabelEncoder()
df["intent_label"] = intent_encoder.fit_transform(df["intent"])
df["task_label"] = task_encoder.fit_transform(df["task"])

os.makedirs(OUTPUT_DIR, exist_ok=True)
with open(f"{OUTPUT_DIR}/intent_mapping.json", "w") as f:
    json.dump({i: label for i, label in enumerate(intent_encoder.classes_)}, f)
with open(f"{OUTPUT_DIR}/task_mapping.json", "w") as f:
    json.dump({i: label for i, label in enumerate(task_encoder.classes_)}, f)

# 5. BIO tagging with alignment
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def bio_tagging(text, slot_json):
    encoding = tokenizer(text, return_offsets_mapping=True, padding='max_length', truncation=True, max_length=MAX_LEN)
    offsets = encoding['offset_mapping']
    word_ids = encoding.word_ids()
    labels = ["O"] * len(offsets)

    for key, value in slot_json.items():
        if not value:
            continue
        value_str = str(value)
        for i, (offset, word_id) in enumerate(zip(offsets, word_ids)):
            if word_id is None or offset is None or offset[0] == 0 and offset[1] == 0:
                continue
            token_text = text[offset[0]:offset[1]]
            if token_text == value_str:
                labels[i] = f"B-{key}"
                for j in range(i+1, len(offsets)):
                    next_offset = offsets[j]
                    if next_offset[0] == 0 and next_offset[1] == 0:
                        break
                    next_token = text[next_offset[0]:next_offset[1]]
                    if next_token in value_str:
                        labels[j] = f"I-{key}"
                    else:
                        break
                break
    return labels

bio_labels = []
slot_labels_set = set()
for _, row in df.iterrows():
    slot_dict = json.loads(row["slots"])
    labels = bio_tagging(row["text"], slot_dict)
    bio_labels.append(labels)
    slot_labels_set.update(labels)

slot_label_list = sorted(list(slot_labels_set))
slot_label_map = {label: i for i, label in enumerate(slot_label_list)}
slot_ignore_index = slot_label_map["O"] # Use 'O' label as ignore index for padding

with open(f"{OUTPUT_DIR}/slot_label_mapping.json", "w") as f:
    json.dump(slot_label_map, f)

# 6. Dataset class
class HRMultiHeadSlotDataset(Dataset):
    def __init__(self, df, bio_labels, tokenizer, max_len=128):
        self.texts = df["text"].tolist()
        self.intent_labels = torch.tensor(df["intent_label"].tolist())
        self.task_labels = torch.tensor(df["task_label"].tolist())
        self.encodings = tokenizer(self.texts, truncation=True, padding="max_length", max_length=max_len, return_tensors="pt")
        self.slot_labels = []
        for i, labels in enumerate(bio_labels):
            label_ids = [slot_label_map.get(l, slot_label_map["O"]) for l in labels]
            label_ids = label_ids[:max_len] + [slot_ignore_index] * (max_len - len(label_ids))
            self.slot_labels.append(torch.tensor(label_ids))

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["intent_labels"] = self.intent_labels[idx]
        item["task_labels"] = self.task_labels[idx]
        item["slot_labels"] = self.slot_labels[idx]
        return item

    def __len__(self):
        return len(self.intent_labels)

# 7. Model definition
class MultiHeadClassifier(torch.nn.Module):
    def __init__(self, base_model_name, num_intents, num_tasks, num_slot_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.encoder.config.hidden_size
        self.intent_head = torch.nn.Linear(hidden_size, num_intents)
        self.task_head = torch.nn.Linear(hidden_size, num_tasks)
        self.slot_head = torch.nn.Linear(hidden_size, num_slot_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None, intent_labels=None, task_labels=None, slot_labels=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.last_hidden_state[:, 0]
        intent_logits = self.intent_head(pooled_output)
        task_logits = self.task_head(pooled_output)
        slot_logits = self.slot_head(outputs.last_hidden_state)

        loss = None
        if intent_labels is not None and task_labels is not None and slot_labels is not None:
            intent_loss = F.cross_entropy(intent_logits, intent_labels)
            task_loss = F.cross_entropy(task_logits, task_labels)
            slot_loss = F.cross_entropy(slot_logits.view(-1, slot_logits.shape[-1]), slot_labels.view(-1), ignore_index=slot_ignore_index)
            loss = intent_loss + task_loss + slot_loss

        return {
            "loss": loss,
            "intent_logits": intent_logits,
            "task_logits": task_logits,
            "slot_logits": slot_logits
        }

# 8. Training setup
model = MultiHeadClassifier(MODEL_NAME, len(intent_encoder.classes_), len(task_encoder.classes_), len(slot_label_list))
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

train_df, val_df, train_bio, val_bio = train_test_split(df, bio_labels, test_size=0.2, random_state=42, stratify=df["intent_label"])
train_dataset = HRMultiHeadSlotDataset(train_df, train_bio, tokenizer, MAX_LEN)
val_dataset = HRMultiHeadSlotDataset(val_df, val_bio, tokenizer, MAX_LEN)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
total_steps = len(train_loader) * EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=int(WARMUP_RATIO * total_steps), num_training_steps=total_steps)

# 9. Training loop
for epoch in range(EPOCHS):
    model.train()
    loop = tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}", leave=False)
    for batch in loop:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs["loss"]
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()
        loop.set_postfix(loss=loss.item())

    # Validation
    model.eval()
    all_intent_preds, all_task_preds = [], []
    all_intent_labels, all_task_labels = [], []
    with torch.no_grad():
        for batch in val_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(input_ids=batch["input_ids"], attention_mask=batch["attention_mask"], token_type_ids=batch.get("token_type_ids"))
            intent_preds = torch.argmax(outputs["intent_logits"], dim=-1)
            task_preds = torch.argmax(outputs["task_logits"], dim=-1)
            all_intent_preds.extend(intent_preds.cpu().numpy())
            all_task_preds.extend(task_preds.cpu().numpy())
            all_intent_labels.extend(batch["intent_labels"].cpu().numpy())
            all_task_labels.extend(batch["task_labels"].cpu().numpy())

    intent_acc = accuracy_score(all_intent_labels, all_intent_preds)
    task_acc = accuracy_score(all_task_labels, all_task_preds)
    intent_f1 = f1_score(all_intent_labels, all_intent_preds, average="weighted")
    task_f1 = f1_score(all_task_labels, all_task_preds, average="weighted")
    print(f"Epoch {epoch+1} | Intent Acc: {intent_acc:.4f} | Task Acc: {task_acc:.4f} | Intent F1: {intent_f1:.4f} | Task F1: {task_f1:.4f}")

# 10. Save model
model.encoder.save_pretrained(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
torch.save(model.state_dict(), f"{OUTPUT_DIR}/multihead_model.pt")
print(f"\n✅ Multi-head model with slot extraction saved at: {OUTPUT_DIR}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

...

Epoch 46 | Intent Acc: 0.9639 | Task Acc: 0.9338 | Intent F1: 0.9635 | Task F1: 0.9337


Epoch 47 | Intent Acc: 0.9656 | Task Acc: 0.9329 | Intent F1: 0.9653 | Task F1: 0.9328


Epoch 48 | Intent Acc: 0.9665 | Task Acc: 0.9355 | Intent F1: 0.9662 | Task F1: 0.9354


Epoch 49 | Intent Acc: 0.9665 | Task Acc: 0.9364 | Intent F1: 0.9662 | Task F1: 0.9362


Epoch 50 | Intent Acc: 0.9665 | Task Acc: 0.9355 | Intent F1: 0.9662 | Task F1: 0.9354

✅ Multi-head model with slot extraction saved at: ./multihead_hr_model


In [ ]:
import os, json, time, re
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModel

# -------------------------
# Paths (update if needed)
# -------------------------
MODEL_DIR = "./multihead_hr_model"   # where training script saved model
MODEL_STATE = os.path.join(MODEL_DIR, "multihead_model.pt")

# -------------------------
# Load tokenizer & label mappings
# -------------------------
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)

with open(os.path.join(MODEL_DIR, "intent_mapping.json")) as f:
    intent_id2label = {int(k): v for k, v in json.load(f).items()}
with open(os.path.join(MODEL_DIR, "task_mapping.json")) as f:
    task_id2label = {int(k): v for k, v in json.load(f).items()}
with open(os.path.join(MODEL_DIR, "slot_label_mapping.json")) as f:
    slot_label_map = json.load(f)
slot_id2label = {v: k for k, v in slot_label_map.items()}
slot_ignore_index = slot_label_map["O"]


# -------------------------
# Model definition (same as training)
# -------------------------
class MultiHeadClassifier(torch.nn.Module):
    def __init__(self, base_model_name, num_intents, num_tasks, num_slot_labels):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(base_model_name)
        hidden_size = self.encoder.config.hidden_size
        self.intent_head = torch.nn.Linear(hidden_size, num_intents)
        self.task_head = torch.nn.Linear(hidden_size, num_tasks)
        self.slot_head = torch.nn.Linear(hidden_size, num_slot_labels)

    def forward(self, input_ids, attention_mask, token_type_ids=None):
        outputs = self.encoder(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        pooled_output = outputs.last_hidden_state[:, 0, :]
        intent_logits = self.intent_head(pooled_output)
        task_logits = self.task_head(pooled_output)
        slot_logits = self.slot_head(outputs.last_hidden_state)
        return intent_logits, task_logits, slot_logits

# -------------------------
# Load trained weights
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = MultiHeadClassifier("sentence-transformers/all-MiniLM-L6-v2",
                            num_intents=len(intent_id2label),
                            num_tasks=len(task_id2label),
                            num_slot_labels=len(slot_id2label))
model.load_state_dict(torch.load(MODEL_STATE, map_location=device))
model.to(device)
model.eval()

# -------------------------
# Helper: decode BIO slots into dictionary
# -------------------------
def decode_slots(text, slot_preds, offsets, tokenizer):
    slots = {}
    current_slot, current_tokens = None, []
    for label_id, (start, end) in zip(slot_preds, offsets):
        if start == 0 and end == 0:  # special or padding
            continue
        label = slot_id2label[label_id]
        if label == "O":
            if current_slot:
                slots[current_slot] = text[current_tokens[0][0]:current_tokens[-1][1]]
                current_slot, current_tokens = None, []
            continue
        if label.startswith("B-"):
            # close previous entity if any
            if current_slot:
                slots[current_slot] = text[current_tokens[0][0]:current_tokens[-1][1]]
            current_slot = label[2:]
            current_tokens = [(start, end)]
        elif label.startswith("I-") and current_slot == label[2:]:
            current_tokens.append((start, end))
        else:
            # mismatch -> reset
            if current_slot:
                slots[current_slot] = text[current_tokens[0][0]:current_tokens[-1][1]]
            current_slot, current_tokens = None, []
    if current_slot:
        slots[current_slot] = text[current_tokens[0][0]:current_tokens[-1][1]]
    return slots

# -------------------------
# Inference function
# -------------------------
def infer(text):
    start_time = time.time()
    enc = tokenizer(text, return_tensors="pt", padding="max_length", truncation=True, max_length=128, return_offsets_mapping=True)
    input_ids = enc["input_ids"].to(device)
    attn = enc["attention_mask"].to(device)
    offsets = enc["offset_mapping"][0].tolist()  # (seq_len, 2)

    with torch.no_grad():
        intent_logits, task_logits, slot_logits = model(input_ids, attn)
        intent_id = torch.argmax(intent_logits, dim=-1).item()
        task_id = torch.argmax(task_logits, dim=-1).item()
        slot_pred_ids = torch.argmax(slot_logits, dim=-1)[0].cpu().tolist()

    # decode slots
    slots = decode_slots(text, slot_pred_ids, offsets, tokenizer)

    end_time = time.time()
    result = {
        "intent": intent_id2label[intent_id],
        "task": task_id2label[task_id],
        "slots": slots,
        "role": "employee"  # role handling depends on dataset; adjust if separate classifier
    }
    print(f"Inference time: {end_time - start_time:.3f} seconds")
    return result

# -------------------------
# Example usage
# -------------------------
if __name__ == "__main__":
    while True:
        q = input("\nEnter query (or 'quit'): ").strip()
        if q.lower() in ("quit","exit"):
            break
        print(infer(q))


Enter query (or 'quit'): what is my leave balance for 2025
Inference time: 0.069 seconds
{'intent': 'leave_balance', 'task': 'leave_balance_summary', 'slots': {'leave_type': 'balance', 'month': '5', 'year': '202'}, 'role': 'employee'}

Enter query (or 'quit'): what is the headcount for the project syncora
Inference time: 0.006 seconds
{'intent': 'project_information', 'task': 'project_deadline', 'slots': {'role': 'for', 'project_name': 'ora', 'perk_type': 'unt', 'department': 'project'}, 'role': 'employee'}

Enter query (or 'quit'): can you guide me through the exit process
Inference time: 0.007 seconds
{'intent': 'exit_process', 'task': 'exit_process_steps', 'slots': {'error_type': 'process', 'year': 'exit'}, 'role': 'employee'}

Enter query (or 'quit'): i am planning to drop my resignation this monthend. What are the necessary steps?
Inference time: 0.009 seconds
{'intent': 'exit_process', 'task': 'exit_process_steps', 'slots': {'report_type': 'i', 'hire_type': 'to', 'update_type': 

In [ ]:
!zip -r /content/multihead_hr_model_improved.zip /content/multihead_hr_model_improved

  adding: content/multihead_hr_model_improved/ (stored 0%)
  adding: content/multihead_hr_model_improved/slot_label_list.json (deflated 78%)
  adding: content/multihead_hr_model_improved/config.json (deflated 47%)
  adding: content/multihead_hr_model_improved/model.safetensors (deflated 8%)
  adding: content/multihead_hr_model_improved/multihead_model_state.pt (deflated 8%)
  adding: content/multihead_hr_model_improved/best_checkpoint.pt (deflated 9%)
  adding: content/multihead_hr_model_improved/vocab.txt (deflated 53%)
  adding: content/multihead_hr_model_improved/intent_mapping.json (deflated 32%)
  adding: content/multihead_hr_model_improved/task_mapping.json (deflated 53%)
  adding: content/multihead_hr_model_improved/slot_label_map.json (deflated 74%)
  adding: content/multihead_hr_model_improved/special_tokens_map.json (deflated 80%)
  adding: content/multihead_hr_model_improved/multihead_model_final.pt (deflated 8%)
  adding: content/multihead_hr_model_improved/tokenizer.json (